In this notebook, we only run the pipeline on chromosome 22 for demonstration purposes.

This allows us to illustrate the full workflow including VCF processing, sample filtering, and IBD analysis, while keeping the runtime manageable.

Before running this notebook, please make sure to have the environment all set and all tools installed.


In [1]:
import os
import re
import time
import pandas as pd
from pathlib import Path
from collections import Counter
import subprocess
import psutil


# PLINK v1.90b7 64-bit (16 Jan 2023)
# !plink --version
# GERMLINE
# !germline --version
# bcftools
# !bcftools --version

First, we need to download required dataset. This process may take as fast as 15 seconds or as slow as a few minutes.

In [2]:
# download ground truth relationship
!wget https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/20140625_related_individuals.txt
# download datasets
!wget https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz
!wget https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz.tbi

!wget https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/supporting/related_samples_vcf/ALL.chr22.phase3_shapeit2_mvncall_integrated_v5_related_samples.20130502.genotypes.vcf.gz
!wget https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/supporting/related_samples_vcf/ALL.chr22.phase3_shapeit2_mvncall_integrated_v5_related_samples.20130502.genotypes.vcf.gz.tbi


--2026-03-09 02:04:11--  https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/20140625_related_individuals.txt
Resolving ftp.1000genomes.ebi.ac.uk (ftp.1000genomes.ebi.ac.uk)... 193.62.193.167
Connecting to ftp.1000genomes.ebi.ac.uk (ftp.1000genomes.ebi.ac.uk)|193.62.193.167|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1168 (1.1K) [text/plain]
Saving to: ‘20140625_related_individuals.txt’

20140625_related_in 100%[===================>]   1.14K  --.-KB/s    in 0s      

2026-03-09 02:04:12 (1.60 GB/s) - ‘20140625_related_individuals.txt’ saved [1168/1168]

--2026-03-09 02:04:13--  https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz
Resolving ftp.1000genomes.ebi.ac.uk (ftp.1000genomes.ebi.ac.uk)... 193.62.193.167
Connecting to ftp.1000genomes.ebi.ac.uk (ftp.1000genomes.ebi.ac.uk)|193.62.193.167|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 205

We can generate a ground-truth relationship file from the metadata (20140625_related_individuals.txt) by parsing the reported relationship types and converting them into pairwise sample relationships for evaluation.

In [3]:
DATA_DIR = Path(".")
GT_TXT = DATA_DIR / "20140625_related_individuals.txt"

def canonical_pair(a: str, b: str):
    return (a, b) if a <= b else (b, a)

def parse_reason(sample_id: str, reason: str):
    reason = str(reason).strip()
    out = []

    # Sibling
    m = re.match(r"^Sibling:(\S+)$", reason)
    if m:
        partner = m.group(1)
        a, b = canonical_pair(sample_id, partner)
        out.append((a, b, "Sibling"))
        return out

    # Parent
    m = re.match(r"^Parent:(\S+)$", reason)
    if m:
        partner = m.group(1)
        a, b = canonical_pair(sample_id, partner)
        out.append((a, b, "Parent"))
        return out

    # Second Order
    m = re.match(r"^Second\s+Order:(\S+)$", reason)
    if m:
        partner = m.group(1)
        a, b = canonical_pair(sample_id, partner)
        out.append((a, b, "SecondOrder"))
        return out

    # Trio
    m = re.match(r"^trio:(\S+),(\S+)$", reason)
    if m:
        p1, p2 = m.group(1), m.group(2)
        a, b = canonical_pair(sample_id, p1)
        out.append((a, b, "Trio"))
        a, b = canonical_pair(sample_id, p2)
        out.append((a, b, "Trio"))
        return out

    return out


# 1) Read file
df = pd.read_csv(GT_TXT, sep=r"\t+", engine="python")

df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist())

# 2) Build pairs
pairs = []
unknown_reasons = Counter()

for _, row in df.iterrows():
    sid = str(row["Sample"]).strip()
    reason = row["Reason for exclusion"]
    extracted = parse_reason(sid, reason)
    if extracted:
        pairs.extend(extracted)
    else:
        unknown_reasons[str(reason).strip()] += 1

pairs_df = (
    pd.DataFrame(pairs, columns=["id1", "id2", "relation"])
    .drop_duplicates()
    .sort_values(["id1", "id2", "relation"])
)

# 3) Write files
pairs_df.to_csv("gt_pairs.tsv", sep="\t", index=False, header=False)

keep_ids = sorted(set(pairs_df["id1"]).union(set(pairs_df["id2"])))
with open("keep.samples", "w") as f:
    for x in keep_ids:
        f.write(x + "\n")

print("gt_pairs.tsv written. n_pairs =", len(pairs_df))
print("keep.samples written. n_samples =", len(keep_ids))
print("\nRelation counts:")
print(pairs_df["relation"].value_counts())

pairs_df.head(10)

Columns: ['Sample', 'Population', 'Gender', 'Reason for exclusion']
gt_pairs.tsv written. n_pairs = 35
keep.samples written. n_samples = 65

Relation counts:
relation
Sibling        11
SecondOrder     9
Trio            8
Parent          7
Name: count, dtype: int64


,id1,id2,relation
0,HG00119,HG00124,SecondOrder
1,HG00501,HG00524,Sibling
2,HG00581,HG00635,Sibling
3,HG00658,HG00702,Sibling
4,HG00731,HG00733,Trio
5,HG00732,HG00733,Trio
6,HG01936,HG01983,SecondOrder
7,HG02024,HG02025,Trio
8,HG02024,HG02026,Trio
9,HG02046,HG02067,Sibling


Here we create "present" versions of the sample list and ground-truth pairs by removing unavailable individuals. This ensures that downstream analysis and evaluation are performed on a consistent set of samples.

In [4]:
CHRS = [22] # using chr22 as a lightweight example for the pipeline

VCF_FULL = "ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"
VCF_REL  = "ALL.chr22.phase3_shapeit2_mvncall_integrated_v5_related_samples.20130502.genotypes.vcf.gz"

KEEP = Path("keep.samples")

keep_ids = [x.strip() for x in KEEP.read_text().splitlines() if x.strip()]
print("keep.samples n =", len(keep_ids))

# sanity: check chr files exist
for c in CHRS:
    f = Path(VCF_FULL.format(c=c))
    r = Path(VCF_REL.format(c=c))
    assert f.exists(), f
    assert r.exists(), r
print(f"Chr{CHRS} VCFs exist.")

missing = {"HG00658"}  

# keep.present
keep = [x.strip() for x in open("keep.samples") if x.strip()]
keep_present = [x for x in keep if x not in missing]

with open("keep.present", "w") as f:
    for x in keep_present:
        f.write(x + "\n")

print("keep.samples:", len(keep))
print("keep.present:", len(keep_present))
print("removed:", sorted(set(keep) - set(keep_present)))

# gt_pairs.present.tsv
gt = pd.read_csv("gt_pairs.tsv", sep="\t", header=None, names=["id1","id2","relation"])
gt_present = gt[~gt["id1"].isin(missing) & ~gt["id2"].isin(missing)].copy()
gt_present.to_csv("gt_pairs.present.tsv", sep="\t", index=False, header=False)

print("gt_pairs.tsv:", len(gt))
print("gt_pairs.present.tsv:", len(gt_present))
gt_present["relation"].value_counts()

keep.samples n = 65
Chr[22] VCFs exist.
keep.samples: 65
keep.present: 64
removed: ['HG00658']
gt_pairs.tsv: 35
gt_pairs.present.tsv: 34


relation
Sibling        10
SecondOrder     9
Trio            8
Parent          7
Name: count, dtype: int64

In [5]:
def vcf_samples(vcf_path: str):
    out = subprocess.check_output(["bcftools", "query", "-l", vcf_path], text=True)
    return set([x.strip() for x in out.splitlines() if x.strip()])

keep_present_set = set([x.strip() for x in open("keep.present") if x.strip()])
full_ids = vcf_samples(VCF_FULL)
rel_ids  = vcf_samples(VCF_REL)

take_from_full = sorted(keep_present_set & full_ids)
take_from_rel  = sorted(keep_present_set & rel_ids)

Path("keep.present.from_full").write_text("\n".join(take_from_full) + ("\n" if take_from_full else ""))
Path("keep.present.from_rel").write_text("\n".join(take_from_rel) + ("\n" if take_from_rel else ""))

print("Take from FULL:", len(take_from_full))
print("Take from REL :", len(take_from_rel))
print("Union:", len(set(take_from_full) | set(take_from_rel)))
print("Expected keep.present:", len(keep_present_set))

merged_vcfs = []
c = 22

OUT_FULL   = f"gt_only.present.chr{c}.from_full.vcf.gz"
OUT_REL    = f"gt_only.present.chr{c}.from_rel.vcf.gz"
OUT_MERGED = f"gt_only.present.chr{c}.vcf.gz"

start = time.time()

# subset FULL/REL
!bcftools view -S keep.present.from_full -Oz -o {OUT_FULL} {VCF_FULL}
!bcftools index -t {OUT_FULL}

!bcftools view -S keep.present.from_rel -Oz -o {OUT_REL} {VCF_REL}
!bcftools index -t {OUT_REL}

# merge (samples are disjoint)
!bcftools merge -Oz -o {OUT_MERGED} {OUT_FULL} {OUT_REL}
!bcftools index -t {OUT_MERGED}

end = time.time()
print("bcftools finished. Elapsed time:", end - start, "seconds")

# quick sanity: sample count
n = int(subprocess.check_output(["bash","-lc", f"bcftools query -l {OUT_MERGED} | wc -l"], text=True).strip())
print(f"chr{c}: merged samples =", n)

merged_vcfs.append(OUT_MERGED)

print("Merged VCFs:", merged_vcfs)

Take from FULL: 33
Take from REL : 31
Union: 64
Expected keep.present: 64
bcftools finished. Elapsed time: 116.40084648132324 seconds
chr22: merged samples = 64
Merged VCFs: ['gt_only.present.chr22.vcf.gz']


In [6]:
keep = [x.strip() for x in open("keep.present") if x.strip()]

with open("keep.present", "w") as f:
    for x in keep:
        f.write(f"{x} {x}\n")

print("written keep.present")

written keep.present


In [16]:
# the result have extremely high P_HAT value for most of the individuals, so we decided to prune first

process = psutil.Process(os.getpid())
start = time.perf_counter()

!source ~/.bashrc \
&& \
plink \
  --vcf ALL.chr22.phase3_shapeit2_mvncall_integrated_v5_related_samples.20130502.genotypes.vcf.gz \
  --keep keep.present \
  --make-bed \
  --out gt_only.present \
&& \
plink \
  --bfile gt_only.present \
  --geno 0.02 \
  --mind 0.02 \
  --maf 0.05 \
  --make-bed \
  --out gt_only.present.qc \
&& \
plink \
  --bfile gt_only.present.qc \
  --indep-pairwise 50 5 0.2 \
  --out gt_only.present.qc.prune \
&& \
plink \
  --bfile gt_only.present.qc \
  --extract gt_only.present.qc.prune.prune.in \
  --make-bed \
  --out gt_only.present.qc.pruned 

end = time.perf_counter()
memory = process.memory_info().rss / (1024**2)

print(f"Pre Runtime: {end-start:.2f} sec")
print(f"Pre Memory usage: {memory:.2f} MB")

single_line_process = psutil.Process(os.getpid())
single_line_start = time.perf_counter()

!source ~/.bashrc \
&& \
plink \
  --bfile gt_only.present.qc.pruned \
  --genome full \
  --out plink_genome.present.qcpruned
single_line_end = time.perf_counter()
single_line_memory = process.memory_info().rss / (1024**2)

print(f"Single Line Runtime: {single_line_end-single_line_start:.2f} sec")
print(f"Single Line  Memory usage: {single_line_memory:.2f} MB")

PLINK v1.90b7 64-bit (16 Jan 2023)             www.cog-genomics.org/plink/1.9/
(C) 2005-2023 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to gt_only.present.log.
Options in effect:
  --keep keep.present
  --make-bed
  --out gt_only.present
  --vcf ALL.chr22.phase3_shapeit2_mvncall_integrated_v5_related_samples.20130502.genotypes.vcf.gz

902879 MB RAM detected; reserving 451439 MB for main workspace.
--vcf: gt_only.present-temporary.bed + gt_only.present-temporary.bim +
gt_only.present-temporary.fam written.
1103547 variants loaded from .bim file.
31 people (0 males, 0 females, 31 ambiguous) loaded from .fam.
Ambiguous sex IDs written to gt_only.present.nosex .
--keep: 31 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 31 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667

In [11]:
gt = pd.read_csv("gt_pairs.present.tsv", sep="\t", header=None, names=["id1","id2","relation"])
gt_pairs = set(zip(gt["id1"].astype(str), gt["id2"].astype(str)))
K = len(gt_pairs)

print("GT present pairs =", K)
gt["relation"].value_counts()

GT present pairs = 34


relation
Sibling        10
SecondOrder     9
Trio            8
Parent          7
Name: count, dtype: int64

In [12]:
gen = pd.read_csv("plink_genome.present.qcpruned.genome", sep=r"\s+", engine="python")
gen.head()

gen.to_csv("plink_genome.present.qcpruned.tsv", sep="\t", index=False)

In [13]:
# use plink (z1+z2) to find relationship (First/second degree)
# gen = pd.read_csv("plink_genome.present.qcpruned.genome", sep=r"\s+", engine="python")

gen["Z1Z2"] = gen["Z1"] + gen["Z2"]

# gen[["IID1","IID2","Z0","Z1","Z2","PI_HAT","Z1Z2"]].head()

def canonical_pair(a, b):
    return (a, b) if a <= b else (b, a)

First_thres = 0.7
Second_thres = 0.4

rows = []
for _, r in gen.iterrows():
    score = float(r["Z1Z2"])
    if score >= First_thres:
        rel = "First"
    elif score >= Second_thres:
        rel = "Second"
    # elif score >= TH_SECOND:
    #     rel = "SecondOrder"
    else:
        continue
    
    a, b = canonical_pair(str(r["IID1"]), str(r["IID2"]))
    rows.append((a, b, rel))

plink_z1z2 = (
    pd.DataFrame(rows, columns=["id1","id2","relation"])
    .drop_duplicates()
    .sort_values(["id1","id2"])
)

plink_z1z2.to_csv("plink_z1z2.present.tsv", sep="\t", index=False, header=False)
print("wrote plink_z1z2.present.tsv")
print("n_pred =", len(plink_z1z2))
plink_z1z2, plink_z1z2.relation.value_counts()


wrote plink_z1z2.present.tsv
n_pred = 2


(       id1      id2 relation
 0  HG02381  HG02388   Second
 1  NA19660  NA19685    First,
 relation
 Second    1
 First     1
 Name: count, dtype: int64)

In [20]:
# GERMLINE

process = psutil.Process(os.getpid())
start = time.perf_counter()

!source ~/.bashrc \
&& \
plink \
  --bfile gt_only.present.qc.pruned \
  --geno 0 \
  --snps-only just-acgt \
  --biallelic-only strict \
  --make-bed \
  --out gt_only.present.qc.pruned.nomissSNP \
&& \
plink \
  --bfile gt_only.present.qc.pruned.nomissSNP \
  --recode \
  --out gt_only.present.qcpruned.nomissSNP_ped \
&& \
germline \
  -input gt_only.present.qcpruned.nomissSNP_ped.ped gt_only.present.qcpruned.nomissSNP_ped.map \
  -output germline.present \
  -bits 128 \
  -min_m 1

end = time.perf_counter()
memory = process.memory_info().rss / (1024**2)

print(f"Runtime: {end-start:.2f} sec")
print(f"Memory usage: {memory:.2f} MB")

PLINK v1.90b7 64-bit (16 Jan 2023)             www.cog-genomics.org/plink/1.9/
(C) 2005-2023 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to gt_only.present.qc.pruned.nomissSNP.log.
Options in effect:
  --bfile gt_only.present.qc.pruned
  --biallelic-only strict
  --geno 0
  --make-bed
  --out gt_only.present.qc.pruned.nomissSNP
  --snps-only just-acgt

902879 MB RAM detected; reserving 451439 MB for main workspace.
90779 out of 103612 variants loaded from .bim file.
31 people (0 males, 0 females, 31 ambiguous) loaded from .fam.
Ambiguous sex IDs written to gt_only.present.qc.pruned.nomissSNP.nosex .
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 31 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rat

In [21]:
MAP = "gt_only.present.qcpruned.nomissSNP_ped.map"
m = pd.read_csv(MAP, sep=r"\s+", header=None, engine="python")

bp = m[3].astype(int)

min_bp = bp.min()
max_bp = bp.max()
L_bp = max_bp - min_bp + 1
L_mb = L_bp / 1e6

MATCH = "germline.present.match"
GT = "gt_pairs.present.tsv"

print("Using:")
print("  MATCH =", MATCH)
print("  GT    =", GT)
print("  L_mb  =", L_mb)

df = pd.read_csv(MATCH, sep=r"\s+", header=None, engine="python")
df.to_csv("germline_match.tsv", sep="\t", index=False, header=False)

# match columns:
# 0=id1, 2=id2, 10=seg_len, 11=unit(MB)
print("Units:", df[11].astype(str).unique())

id1 = df[0].astype(str)
id2 = df[2].astype(str)
seg_len_mb = df[10].astype(float)

def canon(a, b):
    return (a, b) if a <= b else (b, a)

pairs = [canon(a, b) for a, b in zip(id1, id2)]

seg = pd.DataFrame(pairs, columns=["id1","id2"])
seg["seg_len_mb"] = seg_len_mb

agg = (seg.groupby(["id1","id2"])
          .agg(total_ibd_mb=("seg_len_mb","sum"),
               n_seg=("seg_len_mb","size"),
               max_seg_mb=("seg_len_mb","max"))
          .reset_index())

agg["z1z2_hat"] = agg["total_ibd_mb"] / L_mb

agg = agg.sort_values(["z1z2_hat","total_ibd_mb","n_seg"], ascending=False)

agg.to_csv("germline_pairs_norm.present.tsv", sep="\t", index=False)
print("wrote germline_pairs_norm.present.tsv, n_pairs =", len(agg))
# agg.head(15)

TH_FIRST = 0.25
TH_SECOND = 0.10

def classify(z):
    if z >= TH_FIRST:
        return "FirstDegree_like"
    elif z >= TH_SECOND:
        return "SecondDegree_like"
    else:
        return "Distant_or_Unrelated"

pred = agg.copy()
pred["relation"] = pred["z1z2_hat"].apply(classify)

print(pred["relation"].value_counts())
pred.head(20)

Using:
  MATCH = germline.present.match
  GT    = gt_pairs.present.tsv
  L_mb  = 35.189572
Units: <StringArray>
['MB']
Length: 1, dtype: str
wrote germline_pairs_norm.present.tsv, n_pairs = 1
relation
Distant_or_Unrelated    1
Name: count, dtype: int64


,id1,id2,total_ibd_mb,n_seg,max_seg_mb,z1z2_hat,relation
0,HG00635,HG02388,1.036,1,1.036,0.029441,Distant_or_Unrelated


In [18]:
!ls -lh {MATCH}
!wc -l {MATCH}
!head -5 {MATCH}
print(MATCH)

-rw-r--r-- 1 jovyan users 0 Mar  9 02:15 germline.present.match
0 germline.present.match
germline.present.match


In [19]:
m

,0,1,2,3
0,22,.,0,16051249
1,22,.,0,16051453
2,22,.,0,16052080
3,22,.,0,16052962
4,22,.,0,16052986
...,...,...,...,...
90774,22,.,0,51235979
90775,22,.,0,51237063
90776,22,.,0,51237364
90777,22,.,0,51237712
